# 🛒 Zepto Inventory — Exploratory Data Analysis

**Dataset:** 3,732 SKUs scraped from Zepto (India's quick-commerce platform)  
**Source:** [Kaggle — Zepto Inventory Dataset](https://www.kaggle.com/datasets/palvinder2006/zepto-inventory-dataset)  
**Author:** Aditya

---

### Notebook Goals
1. Load and understand the raw dataset
2. Perform thorough EDA with visualisations
3. Clean the data (handle zeros, convert paise → ₹)
4. Surface key business insights through charts
5. Export a clean dataset for ML models

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Styling
plt.style.use('dark_background')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.facecolor': '#0f0f13',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

print('✅ Libraries loaded')

In [ ]:
# ── 1. Load Raw Data ───────────────────────────────────────────────────────
df_raw = pd.read_csv('../data/zepto_v2.csv', encoding='utf-8-sig')
df_raw.columns = df_raw.columns.str.strip()

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# ── 2. Basic Info & Data Types ─────────────────────────────────────────────
df_raw.info()

In [ ]:
# ── 3. Summary Statistics ──────────────────────────────────────────────────
df_raw.describe().round(2)

In [ ]:
# ── 4. Null Value Audit ────────────────────────────────────────────────────
null_report = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing %':     (df_raw.isnull().mean() * 100).round(2)
})
null_report[null_report['Missing Count'] > 0]

In [ ]:
# ── 5. Category Distribution ───────────────────────────────────────────────
cat_counts = df_raw['Category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(cat_counts.index, cat_counts.values,
               color=plt.cm.viridis(np.linspace(0.2, 0.9, len(cat_counts))))
ax.set_xlabel('Number of SKUs')
ax.set_title('SKU Count by Product Category')
for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, color='#aaa')
plt.tight_layout()
plt.savefig('../outputs/eda_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Stock Status ────────────────────────────────────────────────────────
oos = df_raw['outOfStock'].value_counts()
print(f'In Stock:     {oos.get(False, 0):,} ({oos.get(False, 0)/len(df_raw):.1%})')
print(f'Out of Stock: {oos.get(True, 0):,} ({oos.get(True, 0)/len(df_raw):.1%})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Pie
ax1.pie(
    oos.values, labels=['In Stock', 'Out of Stock'],
    colors=['#6bcb77', '#ff6b6b'], autopct='%1.1f%%',
    startangle=90, wedgeprops=dict(edgecolor='#0f0f13', linewidth=2)
)
ax1.set_title('Overall Stock Status')

# Stacked bar by category
cat_oos = df_raw.groupby(['Category', 'outOfStock']).size().unstack(fill_value=0)
cat_oos.columns = ['In Stock', 'Out of Stock']
cat_oos.div(cat_oos.sum(axis=1), axis=0).mul(100).plot(
    kind='bar', stacked=True, ax=ax2,
    color=['#6bcb77', '#ff6b6b'], edgecolor='none'
)
ax2.set_title('Stock Status % by Category')
ax2.set_ylabel('Percentage (%)')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=35)
ax2.legend(loc='upper right', facecolor='#222')

plt.tight_layout()
plt.savefig('../outputs/eda_stock_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7. DATA CLEANING ───────────────────────────────────────────────────────
df = df_raw.copy()

# Remove zero-price records
zero_price = df[(df['mrp'] == 0) | (df['discountedSellingPrice'] == 0)]
print(f'Zero-price records removed: {len(zero_price)}')
df = df[(df['mrp'] > 0) & (df['discountedSellingPrice'] > 0)].copy()

# Convert paise → rupees
df['mrp']                    = df['mrp'] / 100.0
df['discountedSellingPrice'] = df['discountedSellingPrice'] / 100.0

# Engineer new features
df['revenue_proxy']  = df['discountedSellingPrice'] * df['availableQuantity']
df['savings_rs']     = (df['mrp'] - df['discountedSellingPrice']).round(2)
df['price_per_gram'] = (df['discountedSellingPrice'] / df['weightInGms'].replace(0, np.nan)).round(4)
df['weight_segment'] = pd.cut(
    df['weightInGms'],
    bins=[0, 999, 4999, 1e9],
    labels=['Low (< 1 kg)', 'Medium (1–5 kg)', 'Bulk (5 kg+)']
)

print(f'Clean dataset: {df.shape[0]:,} SKUs | {df["Category"].nunique()} categories')
df.head(3)

In [ ]:
# ── 8. MRP Distribution ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
ax1.hist(df['mrp'], bins=50, color='#667eea', edgecolor='none', alpha=0.85)
ax1.set_title('MRP Distribution (all products)')
ax1.set_xlabel('MRP (₹)')
ax1.set_ylabel('Count')
ax1.axvline(df['mrp'].median(), color='#ff6b6b', lw=2, linestyle='--',
            label=f'Median ₹{df["mrp"].median():.0f}')
ax1.legend(facecolor='#222')

# By category (box plot)
cat_order = df.groupby('Category')['mrp'].median().sort_values(ascending=False).index
df_box = df[df['mrp'] <= df['mrp'].quantile(0.95)]  # remove extreme outliers for viz
ax2.boxplot(
    [df_box[df_box['Category'] == c]['mrp'].values for c in cat_order],
    vert=False, patch_artist=True,
    boxprops=dict(facecolor='#667eea', alpha=0.7),
    medianprops=dict(color='#ff6b6b', lw=2),
    whiskerprops=dict(color='#aaa'),
    capprops=dict(color='#aaa'),
    flierprops=dict(marker='.', color='#555', markersize=4),
)
ax2.set_yticklabels(cat_order, fontsize=8)
ax2.set_title('MRP Distribution by Category (95th pctl cap)')
ax2.set_xlabel('MRP (₹)')

plt.tight_layout()
plt.savefig('../outputs/eda_mrp_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9. Discount Analysis ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Discount distribution
axes[0].hist(df['discountPercent'], bins=40, color='#4ECDC4', alpha=0.85, edgecolor='none')
axes[0].set_title('Discount % Distribution')
axes[0].set_xlabel('Discount %')
axes[0].set_ylabel('Count')

# Avg discount by category
avg_disc = df.groupby('Category')['discountPercent'].mean().sort_values(ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(avg_disc)))
axes[1].barh(avg_disc.index, avg_disc.values, color=colors)
axes[1].set_title('Avg Discount % by Category')
axes[1].set_xlabel('Avg Discount %')
for i, v in enumerate(avg_disc.values):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=8, color='#aaa')

# Discount vs MRP scatter
sample = df.sample(min(500, len(df)), random_state=42)
scatter = axes[2].scatter(
    sample['mrp'], sample['discountPercent'],
    c=sample['revenue_proxy'], cmap='viridis',
    alpha=0.6, s=25
)
plt.colorbar(scatter, ax=axes[2], label='Revenue Proxy')
axes[2].set_title('MRP vs Discount % (coloured by Revenue)')
axes[2].set_xlabel('MRP (₹)')
axes[2].set_ylabel('Discount %')

plt.tight_layout()
plt.savefig('../outputs/eda_discount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10. Revenue Potential by Category ─────────────────────────────────────
cat_revenue = (
    df.groupby('Category')
    .agg(
        estimated_revenue=('revenue_proxy', 'sum'),
        total_skus=('name', 'count'),
        avg_discount=('discountPercent', 'mean'),
        stockout_rate=('outOfStock', 'mean'),
    )
    .sort_values('estimated_revenue', ascending=False)
    .round(2)
)

print('Category Health Scorecard:')
print(cat_revenue.to_string())

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    cat_revenue.index,
    cat_revenue['estimated_revenue'] / 1e6,
    color=plt.cm.plasma(np.linspace(0.1, 0.85, len(cat_revenue))),
)
ax.set_title('Estimated Revenue Potential by Category (₹ Millions)')
ax.set_ylabel('Estimated Revenue (₹M)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=35)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
            f'₹{h:.1f}M', ha='center', fontsize=8, color='#ccc')
plt.tight_layout()
plt.savefig('../outputs/eda_category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11. Correlation Heatmap ────────────────────────────────────────────────
num_cols = ['mrp', 'discountPercent', 'discountedSellingPrice',
            'availableQuantity', 'weightInGms', 'savings_rs', 'revenue_proxy']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../outputs/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 12. Key Business Questions (SQL → Python) ──────────────────────────────

print('═' * 60)
print('TOP 10 BEST-VALUE PRODUCTS (by Discount %)')
print('═' * 60)
top_disc = (
    df.drop_duplicates('name')
    .nlargest(10, 'discountPercent')
    [['name', 'Category', 'mrp', 'discountPercent']]
)
print(top_disc.to_string(index=False))

print('\n' + '═' * 60)
print('BEST PRICE-PER-GRAM PRODUCTS (value for money)')
print('═' * 60)
best_value = (
    df[df['weightInGms'] >= 100]
    .drop_duplicates('name')
    .nsmallest(10, 'price_per_gram')
    [['name', 'Category', 'weightInGms', 'discountedSellingPrice', 'price_per_gram']]
)
print(best_value.to_string(index=False))

In [ ]:
# ── 13. Export Clean Dataset ───────────────────────────────────────────────
import os
os.makedirs('../outputs', exist_ok=True)

df.to_csv('../outputs/zepto_clean.csv', index=False)
print(f'✅ Clean dataset saved → outputs/zepto_clean.csv ({df.shape[0]:,} rows)')
print('\n📊 EDA Complete! Key Findings:')
print(f'  • Total clean SKUs: {len(df):,}')
print(f'  • Stockout rate: {df["outOfStock"].mean():.1%}')
print(f'  • Avg MRP: ₹{df["mrp"].mean():.2f}')
print(f'  • Avg Discount: {df["discountPercent"].mean():.1f}%')
print(f'  • Top revenue category: {df.groupby("Category")["revenue_proxy"].sum().idxmax()}')
print(f'  • Highest avg discount: {df.groupby("Category")["discountPercent"].mean().idxmax()}')